## Wolfram Alpha tool
Allows our AI agent to access Wolfram Alpha’s computational engine to solve complex math, physics, and factual queries.
### Use Cases
- **Mathematical Solving**: Calculating integrals, solving differential equations, or performing matrix manipulations.
- **Scientific Queries**: Retrieving physical constants, chemical properties, or astronomical data.
- **Factual Data**: Accessing real-time demographics, economic indicators, or unit conversions. 

Instructions: https://docs.langchain.com/oss/python/integrations/tools/wolfram_alpha

In [31]:
!uv add wolframalpha

Resolved 224 packages in 2ms
Audited 213 packages in 7ms


In [ ]:
import os
from dotenv import load_dotenv
from typing import List, Any, Optional, Dict
from sidekick import State
from langchain_community.utilities.wolfram_alpha import WolframAlphaAPIWrapper
from langchain_community.tools.wolfram_alpha.tool import WolframAlphaQueryRun

load_dotenv(override=True)

wolfram_token = os.environ["WOLFRAM_ALPHA_APPID"]


In [ ]:
async def wolfram_tool():
    wolfram = WolframAlphaAPIWrapper()
    wolfram_tool = WolframAlphaQueryRun(api_wrapper=wolfram)
    return wolfram_tool

wolfram = await wolfram_tool()
print(wolfram.run("What is 2x+5 = -3x + 7?"))

def wolfram_knowledge_worker(state: State) -> Dict[str, Any]:
    # Get the user's query
    query = state["input"]

    # Use the Wolfram Alpha tool to get the answer
    answer = wolfram.run(query)
    return {"answer": [answer]}

Assumption: 2 x + 5 = -3 x + 7 
Answer: x = 2/5


## Planning Agent
This agent will plan the sequence of events when tackling the user's query.
It will throw back 3 questions to the user to further seek clarification then formulate the final query that it will delegate to worker agent and the scientific agent depending on the nature of the question.

In [ ]:
from sidekick import State
from langchain_core.messages import SystemMessage, HumanMessage

planner_llm = ChatOpenAI(model="gpt-4.1-mini")
planner_llm_with_tools = planner_llm.bind_tools(self.tools)

# Schema for structured output to use as routing logic
class Route(BaseModel):
    step: Literal["general", "scientific", "math"] = Field(
        None, description="The next step in the routing process"
    )

def planner(self, state: State) -> State:
    system_message = f"""You are a helpful assistant that receives the user's query and plans the sequence of events 
    to tackle the query.
    You will first throw back 3 questions to the user to further seek clarification. The questions should be based 
    on the user's query and should be designed to help you understand the user's query better.
    After, you will then formulate the final query that it will delegate to worker agent or the scientific agent 
    depending on the nature of the question.
    Route the input to general, scientific, or math based on the user's query.

    This is the success criteria:
    {state["success_criteria"]}

    Once you are finished, reply with the final answer, and don't ask a question; simply reply with the answer.
    """

    if state.get("feedback_on_work"):
        system_message += f"""
            Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
            Here is the feedback on why this was rejected:
            {state["feedback_on_work"]}
            With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user.
            """
    
    # Add in the system message
    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True

    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages

    # Augment the LLM with schema for structured output
    router = planner_llm.with_structured_output(Route)

    # Run the augmented LLM with structured output to serve as routing logic
    decision = router.invoke(
        [
            SystemMessage(content="Route the input to general, scientific, or math based on the user's request."),
            HumanMessage(content=messages),
        ]
    )

    return {"decision": decision.step}

In [ ]:
# Conditional edge function to route to the appropriate node
def route_decision(state: State):
    # Return the node name you want to visit next
    if state["decision"] == "general":
        return "general"
    elif state["decision"] == "scientific":
        return "scientific"
    elif state["decision"] == "math":
        return "math"
    else:
        return "general"